<a href="https://colab.research.google.com/github/saadnaseem11/ICA/blob/main/week%202/%20price_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Price data audit: `teaching_prices_messy.csv`


In [ ]:
import pandas as pd
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 12)
pd.set_option("display.max_colwidth", 30)

raw = pd.read_csv("teaching_prices_messy.csv")
raw.head()

,date,item,city,price,unit,data_label
0,2026-01-01,Staple A,City 1,101.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA...
1,2026-01-01,Staple A,City 2,107.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA...
2,2026-01-01,Oil B,City 1,298.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA...
3,2026-01-01,Oil B,City 2,304.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA...
4,2026-01-01,Pulse C,City 1,200.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA...


**D1. Shape and types**

In [ ]:
print(raw.shape)
raw.dtypes

(97, 6)


,0
date,object
item,object
city,object
price,object
unit,object
data_label,object


i expected many rows and columns but this shows very little

**D2. Missing values**

In [ ]:
raw.isna().sum()

,0
date,0
item,0
city,0
price,1
unit,0
data_label,0


price shows a missing value

**D3. Category labels**

In [ ]:
print(raw["item"].value_counts(), "\n")
print(raw["city"].value_counts(), "\n")
print(raw["unit"].unique())
print(raw["data_label"].unique())

item
Staple A       24
Oil B          24
Pulse C        24
Sweetener D    24
oil b           1
Name: count, dtype: int64 

city
City 1    49
City 2    48
Name: count, dtype: int64 

['PKR per kg (illustrative)']
['ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OBSERVATIONS']


item has 5 labels instead of 4. cities have different rows.

**D4. Duplicates (whole row, then same date-item-city)**

In [ ]:
print(raw[raw.duplicated(keep=False)][["date", "item", "city", "price"]], "\n")
raw[raw.duplicated(["date", "item", "city"], keep=False)][["date", "item", "city", "price"]]

          date      item    city price
8   2026-01-08  Staple A  City 1  99.5
96  2026-01-08  Staple A  City 1  99.5 



,date,item,city,price
8,2026-01-08,Staple A,City 1,99.5
96,2026-01-08,Staple A,City 1,99.5


rows 8 and 96 are the exact same

**D5. Prices that are not numbers**

In [ ]:
price_num = pd.to_numeric(raw["price"], errors="coerce")   # diagnostic copy only
raw[price_num.isna() & raw["price"].notna()][["date", "item", "city", "price"]]

,date,item,city,price
17,2026-01-15,Staple A,City 2,"1,250"


the comma in price makes it text

**D6. Ranges per item (on the diagnostic numeric copy)**

In [ ]:
diag = raw.assign(price_num=price_num)
print(diag.groupby("item")["price_num"].describe().round(1), "\n")
diag.sort_values("price_num", ascending=False).head(3)[["date", "item", "city", "price"]]

             count   mean     std    min    25%    50%    75%     max
item                                                                 
Oil B         24.0  324.8    14.5  298.0  314.8  324.0  337.0   350.0
Pulse C       23.0  205.9     3.9  200.0  202.5  206.0  208.5   213.5
Staple A      23.0  110.9     6.7   99.5  106.0  112.5  115.2   122.0
Sweetener D   24.0  576.8  2006.9  149.0  161.0  167.8  174.9  9999.0
oil b          1.0  108.5     NaN  108.5  108.5  108.5  108.5   108.5 



,date,item,city,price
30,2026-01-22,Sweetener D,City 1,9999
91,2026-03-19,Oil B,City 2,350.0
83,2026-03-12,Oil B,City 2,344.0


sweetener d might be an outlier as it's price is v different from others.

**D7. Expected counts: every date × item × city should appear exactly once**
12 weeks × 4 items × 2 cities = 96 rows. Look for 0s and 2s.

In [ ]:
print("weeks:", raw["date"].nunique())
ct = pd.crosstab(raw["date"], [raw["item"], raw["city"]])
ct

weeks: 12


item        Oil B        Pulse C        Staple A        Sweetener D         oil b
city       City 1 City 2  City 1 City 2   City 1 City 2      City 1 City 2 City 1
date                                                                             
2026-01-01      1      1       1      1        1      1           1      1      0
2026-01-08      1      1       1      1        2      1           1      1      0
2026-01-15      1      1       1      1        1      1           1      1      0
2026-01-22      1      1       1      1        1      1           1      1      0
2026-01-29      1      1       1      1        1      1           1      1      0
2026-02-05      1      1       1      1        0      1           1      1      1
2026-02-12      1      1       1      1        1      1           1      1      0
2026-02-19      1      1       1      1        1      1           1      1      0
2026-02-26      1      1       1      1        1      1           1      1      0
2026-03-05      1      1       1      1        1      1           1      1      0
2026-03-12      1      1       1      1        1      1           1      1      0
2026-03-19      1      1       1      1        1      1           1      1      0

staple a shows errors in dates.

In [ ]:
# the same check by item-city only. Looks fine (12 each), because the duplicate hides the gap
raw.groupby(["item", "city"]).size()

item         city  
Oil B        City 1    12
             City 2    12
Pulse C      City 1    12
             City 2    12
Staple A     City 1    12
             City 2    12
Sweetener D  City 1    12
             City 2    12
oil b        City 1     1
dtype: int64

the 2 on 01-08 covers up for 02-05

**D8. Context for the suspect rows**

In [ ]:
# what was recorded on 2026-02-05?
print(raw[raw["date"] == "2026-02-05"][["date", "item", "city", "price"]], "\n")

# City 2 minus City 1, per date and item: a steady gap makes odd values stand out
wide = diag.pivot_table(index=["date", "item"], columns="city", values="price_num")
wide["gap"] = wide["City 2"] - wide["City 1"]
wide[wide["gap"] != 6]

          date         item    city  price
40  2026-02-05        oil b  City 1  108.5
41  2026-02-05     Staple A  City 2  114.5
42  2026-02-05        Oil B  City 1  318.0
43  2026-02-05        Oil B  City 2  324.0
44  2026-02-05      Pulse C  City 1  202.5
45  2026-02-05      Pulse C  City 2  208.5
46  2026-02-05  Sweetener D  City 1  161.5
47  2026-02-05  Sweetener D  City 2  167.5 



city                    City 1  City 2     gap
date       item                               
2026-01-01 Pulse C       200.0     NaN     NaN
2026-01-15 Staple A      103.0     NaN     NaN
2026-01-22 Sweetener D  9999.0   163.5 -9835.5
2026-02-05 Staple A        NaN   114.5     NaN
           oil b         108.5     NaN     NaN

oil b row is really staple a city 1 mislabelled

**Cleaning log**: write it here (problem · code that revealed it · decision · reason), then commit:
```
git add -A
git commit -m "attempt1"
```

## Cleaning log

| # | Problem | Code that showed it | Decision | Reason |
|---|---|---|---|---|
| 1 | Exact duplicate row: 2026-01-08, Staple A, City 1, 99.5 (rows 8 and 96) | D4 `duplicated(keep=False)` | Exclude second copy | Identical in every column; counts one price twice |
| 2 | `oil b` on 2026-02-05, City 1, 108.5 | D3 `value_counts()`, D7 crosstab, D8 | Investigate → correct to Staple A, flag | Oil B City 1 already has 318 that week; Staple A City 1 is the only missing slot; 108.5 = Staple A City 2 (114.5) − 6, the same gap as every other week. Only fixing capitals would create a second Oil B row |
| 3 | Price "1,250" (2026-01-15, Staple A, City 2) makes price text | D1 `dtypes`, D5 `to_numeric` | Flag, set missing | ~11× nearby prices (103 in City 1, 105.5 and 112.5 in the weeks either side); real value unknown, so I won't guess |
| 4 | Price 9999 (2026-01-22, Sweetener D, City 1) | D6 `describe()` max 9999, std 2006.9 | Flag, set missing | Common missing-value code; nearby prices 153–162 |
| 5 | Blank price (2026-01-01, Pulse C, City 2) | D2 `isna().sum()` | Retain row, leave missing, flag | Don't make up data; the row shows a price was due |
| 6 | `date` stored as text | D1 `dtypes` | Correct to datetime | Needed to sort and work with weeks |
| 7 | `unit` and `data_label` are the same in every row | D3 `unique()` | Retain | `data_label` says the data is illustrative, not real PBS data; dropping it loses that warning |

**Expected after cleaning:** 96 rows, 24 per item, 12 per item-city, 3 missing prices (all flagged).

In [ ]:
# 1. exact duplicate row -> exclude the second copy
mine = raw.drop_duplicates().copy()
mine["flag"] = ""

# 2. 'oil b' on 2026-02-05 City 1 -> correct to Staple A (evidence: D3, D7, D8)
m = (mine["item"] == "oil b") & (mine["date"] == "2026-02-05") & (mine["city"] == "City 1")
mine.loc[m, "item"] = "Staple A"
mine.loc[m, "flag"] = "item relabelled from 'oil b' (evidence: D7/D8)"

# 3. '1,250' -> set missing and flag (true value unknown; 1250 is ~11x its neighbours)
m = mine["price"] == "1,250"
mine.loc[m, "flag"] = "source price '1,250' implausible; set missing"
mine.loc[m, "price"] = None
mine["price"] = pd.to_numeric(mine["price"])   # no errors='coerce': raises if other text is hiding

# 4. 9999 -> missing-value code, set missing and flag
m = mine["price"] == 9999
mine.loc[m, "flag"] = "source price 9999 (missing-value code); set missing"
mine.loc[m, "price"] = float("nan")

# 5. blank price -> retain row, leave missing, flag
mine.loc[mine["price"].isna() & (mine["flag"] == ""), "flag"] = "price missing in source"

# 6. dates as real dates; sort
mine["date"] = pd.to_datetime(mine["date"])
mine = mine.sort_values(["date", "item", "city"])

mine[mine["flag"] != ""][["date", "item", "city", "price", "flag"]]

,date,item,city,price,flag
5,2026-01-01,Pulse C,City 2,NaN,price missing in source
17,2026-01-15,Staple A,City 2,NaN,"source price '1,250' impla..."
30,2026-01-22,Sweetener D,City 1,NaN,source price 9999 (missing...
40,2026-02-05,Staple A,City 1,108.5,item relabelled from 'oil ...


Applied my log with the help of AI. 4 rows are flagged.

## evidence for the trial
- `snapshot(df)`: rows, missing counts, labels, `describe()` per item. Run it **after every AI line**.
- `audit(before, after)`: which original rows were dropped/added and which labels or numbers changed.
- `checks(df)`: does the result match what you expect?

In [ ]:
def snapshot(df, name=""):
    p = pd.to_numeric(df["price"], errors="coerce")
    hidden = int((p.isna() & df["price"].notna()).sum())
    print(f"=== {name}  rows={len(df)}  price dtype={df['price'].dtype}"
          + (f"  ({hidden} text value(s) hidden in describe)" if hidden else ""))
    print("missing:", df.isna().sum().to_dict())
    print("item counts:", df["item"].value_counts().to_dict())
    print(df.assign(price=p).groupby("item")["price"].describe()[["count", "mean", "min", "max"]].round(1), "\n")


def _same(b, a):
    if pd.isna(b) and pd.isna(a):
        return True
    if pd.isna(b) or pd.isna(a):
        return False
    if isinstance(a, pd.Timestamp):
        a = a.strftime("%Y-%m-%d")
    if isinstance(b, pd.Timestamp):
        b = b.strftime("%Y-%m-%d")
    try:
        return float(str(b)) == float(str(a))
    except ValueError:
        return str(b) == str(a)


def audit(before, after, cols=("date", "item", "city", "price")):
    dropped = before.index.difference(after.index)
    added = after.index.difference(before.index)
    print(f"rows {len(before)} -> {len(after)} | dropped {len(dropped)} | extra {len(added)}")
    if len(dropped):
        print("\nDROPPED:\n", before.loc[dropped, list(cols)])
    if len(added):
        print("\nEXTRA (in 'after' only):\n", after.loc[added, [c for c in cols if c in after]])
    common = before.index.intersection(after.index)
    out = []
    for c in cols:
        if c not in after.columns:
            print(f"\nCOLUMN REMOVED: {c}")
            continue
        for i in common:
            b, a = before.at[i, c], after.at[i, c]
            if not _same(b, a):
                out.append((i, c, b, a))
    for c in set(before.columns) - set(after.columns) - set(cols):
        print(f"COLUMN REMOVED: {c}")
    changes = pd.DataFrame(out, columns=["row", "column", "before", "after"])
    print(f"\nCHANGED VALUES: {len(changes)}")
    return changes


def checks(df):
    p = pd.to_numeric(df["price"], errors="coerce")
    res = {
        "96 rows (12 weeks x 4 items x 2 cities)": len(df) == 96,
        "exactly 4 item labels": df["item"].nunique() == 4,
        "every item appears 24 times": (df["item"].value_counts() == 24).all(),
        "every item-city appears 12 times": (df.groupby(["item", "city"]).size() == 12).all(),
        "no duplicate date-item-city": not df.duplicated(["date", "item", "city"]).any(),
        "price is numeric": pd.api.types.is_numeric_dtype(df["price"]),
        "no 9999 or 1250 left as prices": not p.isin([9999, 1250]).any(),
        "data_label column kept": "data_label" in df.columns,
    }
    return pd.Series(res, name="pass")

In [ ]:
snapshot(raw, "raw")
snapshot(mine, "mine")
checks(mine)

=== raw  rows=97  price dtype=object  (1 text value(s) hidden in describe)
missing: {'date': 0, 'item': 0, 'city': 0, 'price': 1, 'unit': 0, 'data_label': 0}
item counts: {'Staple A': 24, 'Oil B': 24, 'Pulse C': 24, 'Sweetener D': 24, 'oil b': 1}
             count   mean    min     max
item                                    
Oil B         24.0  324.8  298.0   350.0
Pulse C       23.0  205.9  200.0   213.5
Staple A      23.0  110.9   99.5   122.0
Sweetener D   24.0  576.8  149.0  9999.0
oil b          1.0  108.5  108.5   108.5 

=== mine  rows=96  price dtype=float64
missing: {'date': 0, 'item': 0, 'city': 0, 'price': 3, 'unit': 0, 'data_label': 0, 'flag': 0}
item counts: {'Oil B': 24, 'Pulse C': 24, 'Staple A': 24, 'Sweetener D': 24}
             count   mean    min    max
item                                   
Oil B         24.0  324.8  298.0  350.0
Pulse C       23.0  205.9  200.0  213.5
Staple A      23.0  111.3   99.5  122.0
Sweetener D   23.0  167.2  149.0  184.5 



,pass
96 rows (12 weeks x 4 items x 2 cities),True
exactly 4 item labels,True
every item appears 24 times,True
every item-city appears 12 times,True
no duplicate date-item-city,True
price is numeric,True
no 9999 or 1250 left as prices,True
data_label column kept,True


## The Cleaning Code Court

In [ ]:
ai = raw.copy()
snapshot(ai, "before AI code")

In [ ]:
# AI line 1  (paste here)

snapshot(ai, "after line 1")

In [ ]:
# AI line 2  (paste here; add more cells as needed)

snapshot(ai, "after line 2")

**Verdict evidence**

In [ ]:
audit(raw, ai)      # vs the original file: what was dropped or changed?

In [ ]:
checks(ai)          # does every item appear the number of times you expect?

In [ ]:
audit(mine, ai)     # where the AI's result differs from your decisions